# TIP Environment Specifics and Claude Code Installation

Interactive exploration of the EPO Technology Intelligence Platform (TIP) runtime environment.

**Part of EPO Academy Training Material provided by mtc.berlin/depa.tech in March 2026**

Run the cells below to inspect your own container.

## 1. User identity & home directory

TIP uses `jovyan` (Jupyter convention) as the base user. A symlink maps your actual username to it.

In [1]:
import os, subprocess

print(f"whoami:    {os.environ.get('USER', subprocess.getoutput('whoami'))}")
print(f"$HOME:     {os.environ['HOME']}")
print(f"realpath:  {os.path.realpath(os.environ['HOME'])}")
print()
# Show the symlink
!ls -la /home/ | grep -v '^\.\.'

whoami:    arne.krueger
$HOME:     /home/arne.krueger
realpath:  /home/jovyan

total 12
drwxr-xr-x  1 root         root  4096 Jul  3 09:22 .
drwxr-xr-x  1 root         root  4096 Jul  3 09:22 ..
lrwxrwxrwx  1 root         root    12 Jul  3 09:22 arne.krueger -> /home/jovyan
drwx------ 18 arne.krueger users 4096 Jul  3 10:21 jovyan


## 2. Filesystem mounts

Only `/home/jovyan` is persistent (shown as `/` in JupyterLab's file browser).
Everything else — `/opt/conda`, the actual system root — is an **overlay** rebuilt from the container image on every restart.

In [2]:
import pandas as pd
import subprocess

# Parse mount info for key paths
rows = []
for target in ["/home/jovyan", "/home/jovyan/training", "/home/jovyan/.cache", "/opt/conda", "/"]:
    result = subprocess.run(
        ["findmnt", "--target", target, "-n", "-o", "TARGET,SOURCE,FSTYPE"],
        capture_output=True, text=True
    )
    if result.stdout.strip():
        parts = result.stdout.strip().split(None, 2)
        source = parts[1] if len(parts) > 1 else "?"
        fstype = parts[2] if len(parts) > 2 else "?"
        # Shorten kubernetes paths for readability
        if "kubernetes.io" in source:
            source = "EmptyDir (Kubernetes)"
        elif source.startswith("/dev/"):
            source = source.split("[")[0]
        persistent = "Yes" if parts[0] == "/home/jovyan" and fstype != "overlay" else "No"
        if parts[0] == "/home/jovyan":
            persistent = "Yes"
        rows.append({"Mount": parts[0], "Source": source, "Type": fstype, "Persistent": persistent})

df = pd.DataFrame(rows)
df.style.set_caption("Container mount points")

,Mount,Source,Type,Persistent
0,/home/jovyan,/dev/sdi,ext4,Yes
1,/home/jovyan/training,EmptyDir (Kubernetes),ext4,No
2,/home/jovyan/.cache,EmptyDir (Kubernetes),ext4,No
3,/,overlay,overlay,No
4,/,overlay,overlay,No


## 3. Training materials (Git worktree mount)

The `~/training/` directory is mounted from a **Kubernetes EmptyDir** populated via a Git worktree checkout at pod startup. All files are owned by UID 65533, not your user. The directory is read-only.

In [3]:
# Show the mount source (reveals the Git worktree path)
!findmnt --target ~/training -o SOURCE -n

print()

# Prove it's read-only in practice
import tempfile, os
test_path = os.path.expanduser("~/training/.write_test")
try:
    open(test_path, "w").close()
    os.remove(test_path)
    print("training/ is WRITABLE (unexpected!)")
except PermissionError:
    print("training/ is READ-ONLY (as expected)")

print()

# List top-level contents with ownership
!ls -la ~/training/

/dev/sda1[/var/lib/kubelet/pods/bc482334-95f0-4570-bab6-98aeaba5ad5b/volumes/kubernetes.io~empty-dir/training/.worktrees/dffd8464767d510c4ebada6a8a9c2e5e165762cf/training]



training/ is READ-ONLY (as expected)



total 20
drwxr-xr-x  5        65533 users 4096 Jul  3 09:22  .
drwx------ 18 arne.krueger users 4096 Jul  3 10:21  ..
drwxr-sr-x  2        65533 users 4096 Jul  3 09:22 'EP full-text data'
drwxr-sr-x  4        65533 users 4096 Jul  3 09:22 'Patent information others'
drwxr-sr-x  6        65533 users 4096 Jul  3 09:22  PATSTAT


## 4. Startup scripts & dotfile persistence

The init script `/usr/bin/init_juser.sh` runs on **every container start** and overwrites
`.bashrc`, `.profile`, and `.condarc` with EPO defaults. Any customizations in those files are lost.

**Rule of thumb:** Put all shell customizations in `.bash_aliases` — it is never touched by init scripts.

In [4]:
import os, datetime

# Check which dotfiles are overwritten on startup vs. persistent
dotfiles = [
    ".bashrc", ".profile", ".condarc",          # overwritten by init
    ".bash_aliases", ".gitconfig", ".npmrc",     # persistent
    ".ssh", ".claude", ".claude.json",           # persistent
]

# Container start time = .bashrc modification time (since it's always overwritten)
boot_time = os.path.getmtime(os.path.expanduser("~/.bashrc"))
boot_str = datetime.datetime.fromtimestamp(boot_time).strftime("%Y-%m-%d %H:%M:%S")
print(f"Container start time (from .bashrc): {boot_str}\n")

print(f"{'File':<20} {'Modified':<22} {'Overwritten on start?'}")
print("-" * 65)
for f in dotfiles:
    path = os.path.expanduser(f"~/{f}")
    if os.path.exists(path):
        mtime = os.path.getmtime(path)
        mtime_str = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M:%S")
        overwritten = "YES — lost on restart!" if abs(mtime - boot_time) < 60 else "No — persistent"
        print(f"{f:<20} {mtime_str:<22} {overwritten}")
    else:
        print(f"{f:<20} {'(not found)':<22}")

Container start time (from .bashrc): 2026-07-03 09:22:45

File                 Modified               Overwritten on start?
-----------------------------------------------------------------
.bashrc              2026-07-03 09:22:45    YES — lost on restart!
.profile             2026-07-03 09:22:45    YES — lost on restart!
.condarc             2026-07-03 09:22:46    YES — lost on restart!
.bash_aliases        2026-07-03 09:46:07    No — persistent
.gitconfig           2026-07-03 10:10:39    No — persistent
.npmrc               2026-07-03 09:24:42    No — persistent
.ssh                 2026-07-03 10:17:32    No — persistent
.claude              2026-07-03 10:22:28    No — persistent
.claude.json         2026-07-03 10:21:09    No — persistent


## 5. Python environment & TIP library access

The EPO TIP library (`epo.tipdata.patstat`) is installed in the base conda environment at `/opt/conda/...`.
When working inside a project venv, this library is invisible unless you explicitly bridge the gap.

In [5]:
import sys

print(f"Python:       {sys.executable}")
print(f"sys.prefix:   {sys.prefix}")
print(f"base_prefix:  {sys.base_prefix}")
print(f"In a venv:    {sys.prefix != sys.base_prefix}")
print()

# Check if TIP library is available
try:
    import epo.tipdata.patstat as tip
    print(f"epo.tipdata.patstat: {tip.__file__}")
except ImportError as e:
    print(f"epo.tipdata.patstat: NOT AVAILABLE ({e})")
    print()
    print("Fix options:")
    print("  1. python -m venv --system-site-packages .venv")
    print('  2. echo "/opt/conda/lib/python3.12/site-packages" > .venv/lib/python3.12/site-packages/conda.pth')

Python:       /opt/conda/bin/python
sys.prefix:   /opt/conda
base_prefix:  /opt/conda
In a venv:    False



epo.tipdata.patstat: /opt/conda/lib/python3.12/site-packages/epo/tipdata/patstat/__init__.py


## 6. Installing Claude Code persistently

`npm install -g` writes to `/opt/conda/` (overlay) by default — lost on restart.
The fix: redirect npm's global prefix to a persistent directory in your home.

Run the cell below for the **initial install** (one-time only, via npm — no Homebrew available on TIP).
After that, use `claude upgrade` to update.

In [6]:
%%bash
set -e

# 1. Create persistent directory
mkdir -p ~/.npm-global

# 2. Set npm prefix (skip if already configured)
if ! grep -q 'npm-global' ~/.npmrc 2>/dev/null; then
    npm config set prefix ~/.npm-global
    echo "Set npm prefix to ~/.npm-global"
else
    echo "npm prefix already configured — skipping"
fi

# 3. Add to PATH in .bash_aliases (idempotent — skips if already present)
touch ~/.bash_aliases
if ! grep -q 'npm-global' ~/.bash_aliases 2>/dev/null; then
    printf '# Persistent npm global packages (survives container restarts)\nexport PATH="$HOME/.npm-global/bin:$PATH"\n' >> ~/.bash_aliases
    echo "Added PATH entry to ~/.bash_aliases"
else
    echo "PATH entry already in ~/.bash_aliases — skipping"
fi

# 4. Install Claude Code
export PATH="$HOME/.npm-global/bin:$PATH"
npm install -g @anthropic-ai/claude-code

# 5. Verify
echo ""
echo "Installed at: $(which claude)"
claude --version


npm prefix already configured — skipping


PATH entry already in ~/.bash_aliases — skipping



added 7 packages, and changed 2 packages in 3s


Installed at: /home/arne.krueger/.npm-global/bin/claude


2.1.199 (Claude Code)


## 7. Connecting Git & GitHub persistently (SSH)

Git and `gh` are pre-installed but unconfigured. We authenticate with an **SSH key** rather than a token — it's simpler and far more robust: SSH authenticates by your **GitHub account**, so the key reaches every repository your account can access, including private organization repos you own or administer.

> ⚠️ **Why not a token?** A *fine-grained* Personal Access Token is scoped to a single **resource owner**. A token owned by your personal account can never see an **organization's** private repos, no matter how many boxes you tick — GitHub just returns *404 Not Found*. SSH sidesteps this entirely.

Everything needed persists under `/home/jovyan` and is **never overwritten** by the init script:

- **Identity** → `~/.gitconfig`
- **SSH key** → `~/.ssh/id_ed25519` (+ `.pub`)
- **Trusted host** → `~/.ssh/known_hosts`

So you do this **once** and it survives every container restart.

**How it works:** edit `GIT_NAME` / `GIT_EMAIL`, then run the cell. It sets your identity, generates a key if you don't have one, and tests the connection. On the **first run** it prints your **public key** and a link — add that key at [github.com/settings/ssh/new](https://github.com/settings/ssh/new), then **re-run the cell** to confirm. Copy the public key as a **single unbroken line** (it starts with `ssh-ed25519 `; a stray line break is the #1 cause of GitHub's *"key is invalid"*).

In [7]:
import os, subprocess

# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  EDIT THESE — your commit identity                                         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
GIT_NAME  = "herrkrueger"                  # your GitHub username / display name
GIT_EMAIL = "arne.krueger@mtc.berlin"      # the email you commit as
# ─────────────────────────────────────────────────────────────────────────────

# ── 1. Git identity → ~/.gitconfig (persistent, never wiped by the init script)
for key, val in {"user.name": GIT_NAME, "user.email": GIT_EMAIL,
                 "init.defaultBranch": "main", "pull.rebase": "false"}.items():
    subprocess.run(["git", "config", "--global", key, val], check=True)
print(f"Git identity: {GIT_NAME} <{GIT_EMAIL}>  (~/.gitconfig)")

# ── 2. SSH key → ~/.ssh (persistent). Auth is by ACCOUNT, so this reaches every
#       repo your GitHub account can see, incl. private org repos you administer.
sshdir = os.path.expanduser("~/.ssh")
os.makedirs(sshdir, mode=0o700, exist_ok=True)
key = os.path.join(sshdir, "id_ed25519")
if not os.path.exists(key):
    subprocess.run(["ssh-keygen", "-t", "ed25519", "-C", GIT_EMAIL,
                    "-f", key, "-N", "", "-q"], check=True)
    print("Generated a new SSH key (~/.ssh/id_ed25519).")
else:
    print("SSH key already present (~/.ssh/id_ed25519).")

# ── 3. Trust github.com's host key (avoids the interactive prompt on first use)
known = os.path.join(sshdir, "known_hosts")
scan = subprocess.run(["ssh-keyscan", "-t", "ed25519,rsa", "github.com"],
                      capture_output=True, text=True)
seen = open(known).read() if os.path.exists(known) else ""
with open(known, "a") as f:
    for line in scan.stdout.splitlines():
        if line and line not in seen:
            f.write(line + "\n")

# ── 4. Test the connection ───────────────────────────────────────────────────
test = subprocess.run(["ssh", "-T", "-o", "StrictHostKeyChecking=accept-new",
                       "git@github.com"], capture_output=True, text=True)
msg = (test.stdout + test.stderr).strip()

if "successfully authenticated" in msg:
    print(f"\n✅ GitHub SSH works — {msg}")
else:
    print("\n⏳ GitHub does not know this key yet. One-time step:")
    print("   1. Open https://github.com/settings/ssh/new")
    print("   2. Paste the PUBLIC key below (a single unbroken line) and save")
    print("   3. Re-run this cell to verify\n")
    print("   ── your public key ──────────────────────────────────────────")
    print("   " + open(key + ".pub").read().strip())
    print("   ─────────────────────────────────────────────────────────────")


Git identity: herrkrueger <arne.krueger@mtc.berlin>  (~/.gitconfig)
SSH key already present (~/.ssh/id_ed25519).



✅ GitHub SSH works — Hi herrkrueger! You've successfully authenticated, but GitHub does not provide shell access.


## 8. Disk usage overview

Check how much space you're using on your persistent volume vs the ephemeral overlay.

In [8]:
!echo "=== Your persistent volume ===" && df -h /home/jovyan
!echo ""
!echo "=== Overlay (ephemeral) ===" && df -h /
!echo ""
!echo "=== Top directories by size in your home ===" && du -sh ~/*/  2>/dev/null | sort -rh | head -15

=== Your persistent volume ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/sdi         30G  739M   29G   3% /home/jovyan


=== Overlay (ephemeral) ===
Filesystem      Size  Used Avail Use% Mounted on
overlay          95G   33G   62G  35% /


=== Top directories by size in your home ===
175M	/home/arne.krueger/epo-tip4patlibs/
48M	/home/arne.krueger/training/
16K	/home/arne.krueger/lost+found/


## 9. Getting the training material

The exercises for this course live in the **`mtcberlin/epo-tip4patlibs`** repository. The cell below clones it over **SSH** into `~/epo-tip4patlibs` on your persistent volume — or, if it's already there, pulls the latest changes.

Because it uses the SSH key from Section 7, it works for the private org repo without any token. Once cloned, the material appears in JupyterLab's file browser under `epo-tip4patlibs/`.

In [9]:
import os, subprocess

# The training-material repository (edit if you forked it under your own account)
REPO = "mtcberlin/epo-tip4patlibs"

# Clone into your persistent home so it survives container restarts.
DEST = os.path.expanduser(f"~/{REPO.split('/')[1]}")   # ~/epo-tip4patlibs

if os.path.isdir(os.path.join(DEST, ".git")):
    print(f"Already cloned at {DEST} — pulling latest changes...")
    r = subprocess.run(["git", "-C", DEST, "pull", "--ff-only"],
                       capture_output=True, text=True)
    print((r.stdout or r.stderr).strip())
    if r.returncode != 0:
        print("(pull failed — resolve manually if you have local changes)")
else:
    print(f"Cloning {REPO} into {DEST} ...")
    # SSH URL → uses the key from Section 7, so private org repos work too.
    r = subprocess.run(["git", "clone", f"git@github.com:{REPO}.git", DEST],
                       capture_output=True, text=True)
    print((r.stdout or r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError(
            f"Clone failed. Did the SSH test in Section 7 succeed, and does your "
            f"account have access to {REPO}?\n{r.stderr}")
    print("Cloned.")

# Show what we got
if os.path.isdir(DEST):
    entries = sorted(e for e in os.listdir(DEST) if not e.startswith("."))
    print(f"\nContents of {DEST}:")
    for e in entries:
        tag = "/" if os.path.isdir(os.path.join(DEST, e)) else ""
        print(f"  {e}{tag}")


Already cloned at /home/arne.krueger/epo-tip4patlibs — pulling latest changes...


Already up to date.

Contents of /home/arne.krueger/epo-tip4patlibs:
  1_notebooks/
  2_querylib/
  3_claude/
  4_patstat_explorer/
  CLAUDE.md
  README.md
  _bmad/
  _bmad-output/
  context/
  docs/
  exports/
  harmonization/
  ipc-extension/
  setup/
  tests/
